# 🤖 Machine Learning Model: Ticket Spam & Phishing Detection
This notebook builds and evaluates an end-to-end Machine Learning pipeline to classify incoming tickets into:
* **Legitimate Support Ticket (`0`)**
* **Spam / Bot / Phishing (`1`)**

Achieving **~98% Accuracy**, this model acts as the first-line security gatekeeper in the ticket classification platform before tickets reach Category and Priority models.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score

## 1. Load Dataset & Train-Test Split
Loading `ticket_spam_dataset.csv` and splitting into an 80% training set and 20% test set with stratified sampling.

In [2]:
df = pd.read_csv('ticket_spam_dataset.csv')
X = df['text']
y = df['is_spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print(f"Spam ratio in test set: {y_test.mean() * 100:.2f}%")

Training samples: 4457
Testing samples:  1115
Spam ratio in test set: 13.36%


## 2. Model 1: Multinomial Naive Bayes Pipeline (Industry Standard)
Multinomial Naive Bayes combined with TF-IDF is the classic, highly effective gold standard for text spam filtering due to its calibrated posterior probabilities and speed.

In [3]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=4000, stop_words='english', ngram_range=(1, 2))),
    ('nb', MultinomialNB(alpha=0.1))
])

nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)
y_prob_nb = nb_pipeline.predict_proba(X_test)[:, 1]

print("=== Multinomial Naive Bayes Results ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_nb):.4f}\n")
print(classification_report(y_test, y_pred_nb, target_names=['Legitimate', 'Spam']))

=== Multinomial Naive Bayes Results ===
Accuracy:  0.9830
ROC-AUC:   0.9889

              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99       966
        Spam       0.98      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## 3. Model 2: Logistic Regression Pipeline
Training a penalized Logistic Regression classifier for comparison.

In [4]:
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=4000, stop_words='english', ngram_range=(1, 2))),
    ('lr', LogisticRegression(max_iter=500, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Results ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}\n")
print(classification_report(y_test, y_pred_lr, target_names=['Legitimate', 'Spam']))

=== Logistic Regression Results ===
Accuracy:  0.9749
ROC-AUC:   0.9847

              precision    recall  f1-score   support

  Legitimate       0.97      1.00      0.99       966
        Spam       1.00      0.81      0.90       149

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115



## 4. Confusion Matrix Visualization
Evaluating false positives and false negatives to ensure legitimate user tickets are never erroneously discarded.

In [5]:
cm = confusion_matrix(y_test, y_pred_nb)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Spam'],
            yticklabels=['Legitimate', 'Spam'])
plt.title(f'Confusion Matrix (Accuracy: {accuracy_score(y_test, y_pred_nb) * 100:.1f}%)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

## 5. Model Export for Production
Saving the top-performing pipeline to `models/ticket_spam_detector.joblib` so it can be loaded directly by the backend API or dashboard.

In [6]:
os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', 'ticket_spam_detector.joblib')

joblib.dump(nb_pipeline, model_path)
print(f"Model successfully saved to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1024:.2f} KB")

Model successfully saved to: models\ticket_spam_detector.joblib
File size: 273.25 KB


## 6. Live Inference Demonstration
Testing the exported pipeline with both authentic tech support tickets and modern phishing/spam inquiries.

In [7]:
loaded_detector = joblib.load(model_path)

test_tickets = [
    # Legitimate IT Support Tickets
    "PostgreSQL connection timeout in production cluster. DB pool exhausted on primary replica.",
    "Unable to connect to corporate VPN from MacOS Sequoia. WireGuard handshake fails.",
    "Invoice billing discrepancy: charged twice for October annual subscription.",
    
    # Spam / Bot / Phishing Submissions
    "URGENT: Your account credentials have been flagged! Click here to verify http://secure-auth-bank.xyz",
    "Congratulations! You won a $1,000 Walmart Gift Card! Call 0800-888-999 now to claim.",
    "Work from home and earn $5,000 weekly! No experience required! Reply YES for details."
]

print("=== REAL-TIME INFERENCE RESULTS ===\n")
for ticket in test_tickets:
    pred = loaded_detector.predict([ticket])[0]
    prob = loaded_detector.predict_proba([ticket])[0]
    status = "🚨 SPAM / PHISHING" if pred == 1 else "✅ LEGITIMATE TICKET"
    confidence = prob[pred] * 100
    
    print(f"Ticket: {ticket[:70]}...")
    print(f"Result: {status} (Confidence: {confidence:.2f}%)\n")

=== REAL-TIME INFERENCE RESULTS ===

Ticket: PostgreSQL connection timeout in production cluster. DB pool exhausted...
Result: ✅ LEGITIMATE TICKET (Confidence: 95.68%)

Ticket: Unable to connect to corporate VPN from MacOS Sequoia. WireGuard hands...
Result: ✅ LEGITIMATE TICKET (Confidence: 81.09%)

Ticket: Invoice billing discrepancy: charged twice for October annual subscrip...
Result: 🚨 SPAM / PHISHING (Confidence: 84.26%)

Ticket: URGENT: Your account credentials have been flagged! Click here to veri...
Result: 🚨 SPAM / PHISHING (Confidence: 97.38%)

Ticket: Congratulations! You won a $1,000 Walmart Gift Card! Call 0800-888-999...
Result: 🚨 SPAM / PHISHING (Confidence: 99.61%)

Ticket: Work from home and earn $5,000 weekly! No experience required! Reply Y...
Result: 🚨 SPAM / PHISHING (Confidence: 75.01%)

